# 04 · Búsqueda Semántica (Recuperación)
Carga el índice FAISS y recupera los fragmentos más relevantes para una
consulta usando **k dinámico** determinado por un LLM.

### ¿Por qué Groq para el k dinámico?
La selección de cuántos fragmentos recuperar es una decisión **simple y
rápida**: solo requiere evaluar la complejidad de la consulta y retornar
un número. Groq con LLaMA 3.3 70B ofrece latencia ultra-baja (<500 ms),
ideal para esta micro-decisión sin sacrificar calidad.


In [ ]:
# ── Instalar dependencias ──────────────────────────────────────────────────
!pip install -q \
    langchain==0.3.14 \
    langchain-community==0.3.14 \
    langchain-google-genai==2.0.8 \
    langchain-groq==0.2.3 \
    faiss-cpu==1.9.0
print('✓ Dependencias instaladas')


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

BASE_DIR  = '/content/drive/MyDrive/RAG_BioMed'
INDEX_DIR = f'{BASE_DIR}/faiss_index'

GEMINI_KEY   = userdata.get('GEMINI_API_KEY')
GROQ_KEY     = userdata.get('GROQ_API_KEY')
EMBED_MODEL  = 'models/text-embedding-004'
GROQ_MODEL   = 'llama-3.3-70b-versatile'


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

def cargar_vectorstore(index_dir: str, api_key: str) -> FAISS:
    """
    Carga el índice FAISS persistido en Drive.

    Args:
        index_dir : Ruta a la carpeta con el índice FAISS.
        api_key   : Google Gemini API key para inicializar embeddings.

    Returns:
        Instancia FAISS lista para búsqueda.
    """
    embeddings = GoogleGenerativeAIEmbeddings(
        model=EMBED_MODEL,
        google_api_key=api_key
    )
    vs = FAISS.load_local(
        index_dir, embeddings,
        allow_dangerous_deserialization=True
    )
    print(f'✓ Índice cargado — {vs.index.ntotal} vectores')
    return vs

vs = cargar_vectorstore(INDEX_DIR, GEMINI_KEY)


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

def determinar_k(query: str, tipo: str, groq_key: str) -> int:
    """
    Usa Groq LLaMA para determinar dinámicamente el número de fragmentos
    (k) a recuperar según la complejidad de la consulta.

    Rango permitido: [3, 15].

    Args:
        query    : Consulta del usuario.
        tipo     : Categoría ('búsqueda', 'resumen', 'comparación').
        groq_key : Groq API key.

    Returns:
        Entero k en [3, 15].
    """
    llm = ChatGroq(model=GROQ_MODEL, groq_api_key=groq_key, temperature=0.0)
    prompt = ChatPromptTemplate.from_messages([
        ('system', """Decides cuántos fragmentos (k) recuperar de un corpus biomédico.
Guía:
- búsqueda simple      → k entre 3 y 5
- resumen de un paper  → k entre 5 y 8
- comparación          → k entre 6 y 10
- consulta muy amplia  → k entre 10 y 15
Responde SOLO con el número entero, sin texto adicional."""),
        ('human', 'Tipo: {tipo}\nConsulta: {query}')
    ])
    chain = prompt | llm
    resp = chain.invoke({'tipo': tipo, 'query': query})
    try:
        return max(3, min(int(resp.content.strip()), 15))
    except ValueError:
        return 5   # valor por defecto


In [ ]:
from langchain_core.documents import Document

def recuperar(query: str, tipo: str, vectorstore: FAISS,
              groq_key: str) -> list[Document]:
    """
    Recupera fragmentos relevantes con k determinado dinámicamente.
    Añade `similarity_score` a los metadatos para trazabilidad.

    Args:
        query       : Consulta del usuario.
        tipo        : Tipo de consulta para seleccionar k.
        vectorstore : Índice FAISS cargado.
        groq_key    : Groq API key.

    Returns:
        Lista de Documents con metadatos enriquecidos.
    """
    k = determinar_k(query, tipo, groq_key)
    print(f'  k dinámico seleccionado: {k}')

    docs_scores = vectorstore.similarity_search_with_score(query, k=k)
    resultados = []
    for doc, score in docs_scores:
        doc.metadata['similarity_score'] = round(float(score), 4)
        resultados.append(doc)
        m = doc.metadata
        print(f'  [{m["doc_id"]}] p.{m.get("page",0)+1} '
              f'chunk={m["chunk_id"]}  score={score:.4f}')
        print(f'    {doc.page_content[:100]}...')
    return resultados


In [ ]:
# ── Pruebas de recuperación ────────────────────────────────────────────────
pruebas = [
    ('¿Qué redes neuronales predicen interacciones proteína-proteína?', 'búsqueda'),
    ('Resume los avances en diagnóstico de cáncer con IA',             'resumen'),
    ('Compara métodos de ensamble en genómica',                        'comparación'),
]

for query, tipo in pruebas:
    print(f'\n=== QUERY: {query[:70]} ===')
    docs = recuperar(query, tipo, vs, GROQ_KEY)
    print(f'  → {len(docs)} fragmentos recuperados')
